# AI and the Language of DNA: results
Team SouthernAI, CS F407, BITS Pilani Goa

Can a model that reads DNA as a language tell pathogens from harmless bacteria?

In [ ]:
#@title load results
import base64, gzip, io, json, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# all our result csvs + the 2-mer counts of every fragment, gzipped and base64 encoded
data = (
"..."
)
files = json.loads(gzip.decompress(base64.b64decode(data)))
res = {name: pd.read_csv(io.StringIO(txt)) for name, txt in files.items()}
genomes = res['genomes.csv']
print(len(genomes), 'genomes,', len(res['frag_2mer_counts.csv']), 'fragments')

## 1. Data
24 genomes (12 pathogen, 12 harmless), 400 fragments x 1000 bp each. 6 matched pairs, 6 genomes held out for testing.

In [ ]:
pairs = genomes[genomes.pair_group == 'pair']
pd.DataFrame({
    'pathogen': pairs[pairs.label == 1].set_index('genus').organism,
    'harmless relative': pairs[pairs.label == 0].set_index('genus').organism,
})

In [ ]:
g = genomes.sort_values('genome_gc')
plt.figure(figsize=(10, 5.5))
plt.bar(g.organism, g.genome_gc * 100, color=['tab:red' if l == 1 else 'tab:blue' for l in g.label])
plt.xticks(rotation=80, fontsize=7)
plt.ylabel('GC content %')
plt.title('GC content of the 24 genomes (red = pathogen, blue = harmless)')
plt.tight_layout()
plt.show()

## 2. Best accuracy vs random labels
Best of 36 models (BoW / TF-IDF / Word2Vec x 4 classifiers) compared with 200 runs on randomly labelled genomes.

In [ ]:
clf = res['classifiers.csv']
best = clf.test_acc.max()
null = res['null_random_labels.csv'].test_acc

print('best test accuracy:', round(best, 3))
print('random-label runs that did as well or better:', (null >= best).mean())

plt.figure(figsize=(7, 3))
plt.hist(null, bins=25, color='lightgray')
plt.axvline(best, color='red', lw=2, label='our best model')
plt.xlabel('test accuracy with shuffled labels')
plt.ylabel('runs')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(clf.train_acc, clf.test_acc)
plt.plot([0.45, 1], [0.45, 1], 'k--', lw=1, label='test = train')
plt.axhline(0.5, color='gray', ls=':', label='chance')
plt.xlabel('accuracy on training genomes')
plt.ylabel('accuracy on unseen genomes')
plt.title('All 36 models')
plt.legend()
plt.show()

## 3. Final model, trained live
2-mer frequencies + logistic regression, chosen by cross validation on training genomes only.

In [ ]:
d = res['frag_2mer_counts.csv']
words = [a + b for a, b in itertools.product('ACGT', repeat=2)]
X = d[words].values / d[words].values.sum(axis=1, keepdims=True)
y = genomes.label.values[d.g]
gid = genomes.genome_id.values[d.g]
train = d.tr.values == 1
test = ~train

model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced'))
model.fit(X[train], y[train])
p = model.predict_proba(X[test])[:, 1]
print('fragment accuracy on unseen genomes:', round(((p > 0.5) == y[test]).mean(), 3))

# average over fragments to get one score per genome
out = pd.DataFrame({'genome_id': gid[test], 'p_pathogen': p}).groupby('genome_id').mean()
out = out.join(genomes.set_index('genome_id')[['organism', 'genus', 'label']])
out['called'] = (out.p_pathogen > 0.5).astype(int)
out.sort_values('genus')[['organism', 'genus', 'label', 'p_pathogen', 'called']]

In [ ]:
o = out.sort_values(['genus', 'label'])
plt.figure(figsize=(8, 4.5))
plt.bar(o.organism, o.p_pathogen, color=['tab:red' if l == 1 else 'tab:blue' for l in o.label])
plt.axhline(0.5, color='k', ls='--', lw=1)
plt.xticks(rotation=25, ha='right', fontsize=8)
plt.ylabel('mean P(pathogen)')
plt.title('The 6 test genomes (red = actually pathogen, dashed = decision line)')
plt.tight_layout()
plt.show()

## 4. What it actually learned
All 24 genomes scored with their whole genus left out (red = pathogen).

In [ ]:
logo = res['logo.csv'].set_index('model')
print('best leave-one-genus-out: %d/24 correct (chance = 12), p = %.2f'
      % (logo.n_genomes_correct.max(), logo.loc[logo.n_genomes_correct.idxmax(), 'perm_p_acc']))

perm = res['perm_null.csv']
fig, ax = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for a, m in zip(ax, ['GC-only', '2-mer', '3-mer']):
    a.hist((perm[perm.model == m].genome_acc * 24).round(), bins=range(4, 22), color='lightgray')
    a.axvline(logo.loc[m, 'n_genomes_correct'], color='red', lw=2)
    a.set_title(m + ' (red = real labels)')
    a.set_xlabel('genomes correct out of 24')
ax[0].set_ylabel('random-label runs')
plt.tight_layout()
plt.show()

In [ ]:
s = res['genome_scores.csv']
s = s[s.model == '2-mer'].merge(genomes[['genome_id', 'genome_gc']], on='genome_id')
s = s.sort_values('logo_mean_p_pathogen')
colors = ['tab:red' if l == 1 else 'tab:blue' for l in s.label]

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].barh(s.organism, s.logo_mean_p_pathogen, color=colors)
ax[0].axvline(0.5, color='k', ls='--', lw=1)
ax[0].set_xlabel('predicted P(pathogen)')
ax[0].tick_params(axis='y', labelsize=7)
ax[1].scatter(s.genome_gc * 100, s.logo_mean_p_pathogen, c=colors)
ax[1].set_xlabel('genome GC %')
ax[1].set_ylabel('predicted P(pathogen)')
plt.tight_layout()
plt.show()

In [ ]:
probes = res['probes.csv']
names = probes.rep.str.upper() + ' k=' + probes.k.astype(str)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].bar(names, probes.genome_identity_acc_random_split * 100, color='tab:green')
ax[0].axhline(100 / 18, color='k', ls='--', lw=1)
ax[0].set_ylim(0, 100)
ax[0].set_ylabel('accuracy %')
ax[0].set_title('Which of 18 genomes is this fragment from? (dashed = chance)')
ax[1].bar(names, probes.mean_within_pair_AUC, color='tab:orange')
ax[1].axhline(0.5, color='k', ls='--', lw=1)
ax[1].set_ylim(0, 1)
ax[1].set_ylabel('AUC')
ax[1].set_title('Pathogen vs its own relative (dashed = coin flip)')
plt.tight_layout()
plt.show()

## 5. Does word order matter?
Test fragments re-scored after shuffling single letters or 10-letter blocks.

In [ ]:
gr = res['grammar.csv']
gr = gr[gr.model.isin(['3-mer', '4-mer'])].copy()
gr['condition'] = gr.condition.str.split(' (', regex=False).str[0]
pv = gr.pivot(index='condition', columns='model', values='test_fragment_acc')
pv = pv.loc[['original', 'letters shuffled', '10-bp blocks shuffled']]
ax = (pv * 100).plot.bar(rot=0, figsize=(7, 3))
ax.axhline(50, color='k', ls=':', lw=1)
ax.set_ylim(45, 62)
ax.set_ylabel('test accuracy %')
ax.set_xlabel('')
plt.show()

## 6. DNA only the pathogen has
E. coli O157:H7 fragments with <10% of 16-mers found in harmless K-12, scored by a 4-mer model with Escherichia left out.

In [ ]:
e = res['pathogen_specific.csv']
e = e[e.pair == 'Escherichia'].iloc[0]

plt.figure(figsize=(6, 2.5))
plt.barh(['shared with K-12', 'only in O157:H7'], [e.p_path_shared, e.p_path_specific], color=['gray', 'tab:red'])
plt.xlim(0, 1)
plt.xlabel('predicted P(pathogen)')
plt.show()
print('pathogen-only fragments: %.0f%% of O157:H7' % (100 * e.share_pathogen_specific))
print('shared:', round(e.p_path_shared, 2), ' pathogen-only:', round(e.p_path_specific, 2))

## Conclusion
- It learns the organism, not the disease.
- Pathogen genes are a tiny part of a genome.
- 7,200 fragments from 18 genomes = effective sample size of 18.